# 13 Feedback Loop from Accepted LLM Suggestions

This notebook takes accepted suggestion fields, feeds them back into the deterministic canonical engine, and shows which rows became canonical-ready.

In [ ]:
from pathlib import Path
from datetime import datetime
import sys
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
POLICY_PATH = PROJECT_ROOT / 'policy' / 'SCH_fileserver_policy_v2_4.yaml'

from src.canonicalize import load_policy, CanonicalizeConfig
from src.feedback_loop import (
    FeedbackConfig,
    build_feedback_review,
    rerun_canonical_with_feedback,
    feedback_summary,
)

policy = load_policy(POLICY_PATH)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('POLICY_PATH exists =', POLICY_PATH.exists())


In [ ]:
AUTO_ACCEPT_FIELDS = ['description', 'doc_type', 'phase', 'date', 'version', 'status']
MIN_CONFIDENCE = 0.80
REQUIRE_CONTENT_DERIVED_DESCRIPTION = True

def latest_exact(prefix: str) -> Path:
    candidates = sorted(OUTPUT_DIR.glob(f'{prefix}_*.parquet'))
    if not candidates:
        raise FileNotFoundError(f'No parquet outputs found for prefix: {prefix}')
    return candidates[-1]

CANONICAL_PATH = latest_exact('canonical_candidates')
SUGGESTIONS_PATH = latest_exact('llm_suggestions')

print('CANONICAL_PATH =', CANONICAL_PATH.name)
print('SUGGESTIONS_PATH =', SUGGESTIONS_PATH.name)


In [ ]:
canonical = pd.read_parquet(CANONICAL_PATH)
suggestions = pd.read_parquet(SUGGESTIONS_PATH)

feedback_config = FeedbackConfig(
    auto_accept_fields=AUTO_ACCEPT_FIELDS,
    min_confidence=MIN_CONFIDENCE,
    require_content_derived_description=REQUIRE_CONTENT_DERIVED_DESCRIPTION,
)

review = build_feedback_review(canonical, suggestions, config=feedback_config)
print('Rows in feedback review:', len(review))

preview_cols = [
    'relative_path', 'unresolved_fields', 'suggestion_confidence',
    'suggested_description', 'suggested_description_source', 'accept_description',
    'suggested_doc_type', 'accept_doc_type',
    'suggested_phase', 'accept_phase',
    'accepted_field_count', 'has_any_accepted_suggestion',
]
display(review[[c for c in preview_cols if c in review.columns]].head(20))


## Optional manual review

At this point you can export `review` and manually edit any `accept_*` columns before rerunning the next cells.

In [ ]:
review_export_csv = OUTPUT_DIR / 'feedback_review_latest.csv'
review_export_parquet = OUTPUT_DIR / 'feedback_review_latest.parquet'
review.to_csv(review_export_csv, index=False, encoding='utf-8-sig')
review.to_parquet(review_export_parquet, index=False)
print('Saved review exports to:')
print('-', review_export_csv)
print('-', review_export_parquet)


In [ ]:
rerun = rerun_canonical_with_feedback(
    review,
    policy,
    canonical_config=CanonicalizeConfig(),
)
summary = feedback_summary(rerun)
summary


In [ ]:
display(rerun[['relative_path', 'accepted_fields', 'canonical_ready_before', 'canonical_ready_after', 'became_canonical_ready', 'unresolved_fields_before', 'unresolved_fields_after', 'canonical_relative_path_after']].head(30))

display(rerun['canonical_ready_after'].fillna(False).value_counts().rename_axis('canonical_ready_after').reset_index(name='count'))

improved = rerun.loc[rerun['became_canonical_ready'].fillna(False)].copy()
display(improved[['relative_path', 'accepted_fields', 'canonical_relative_path_after', 'canonical_description_after']].head(50))


In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
rerun_csv = OUTPUT_DIR / f'canonical_feedback_rerun_{ts}.csv'
rerun_parquet = OUTPUT_DIR / f'canonical_feedback_rerun_{ts}.parquet'
rerun.to_csv(rerun_csv, index=False, encoding='utf-8-sig')
rerun.to_parquet(rerun_parquet, index=False)
print('Saved rerun outputs:')
print('-', rerun_csv)
print('-', rerun_parquet)
